# Sample Size Calculation for A/B Testing — Complete Solution

**This is the companion solution notebook.** Use it to check your work from the Practice Skeleton, or study the complete implementations, alternate approaches, Monte Carlo verification, and deeper simulations.

All code cells are fully runnable and produce clear printed output.


## Theory: Why Sample Size Matters

Sample size calculation is one of the most important steps **before** running any experiment or A/B test.

### Why it is crucial:

1. **Statistical Power (1 - β)**: Power is the probability of detecting a true effect (rejecting H0 when it is false). Underpowered studies (too small N) have high Type II error rate — you miss real improvements that exist.

2. **Avoid Wasted Resources**: Running an experiment that is too small often leads to inconclusive results (“we didn’t see significance, but maybe we just didn’t have enough data”). You spent time and traffic for nothing.

3. **Avoid Overpowered Studies**: Very large samples can detect tiny effects that are statistically significant but practically meaningless (e.g. 0.1% lift that costs more to implement than it returns).

4. **Ethical & Business Reasons**: In user-facing tests (ads, UX, pricing), you expose real users to variants. Too long = opportunity cost or potential harm; too short = unreliable conclusions.

5. **Reproducibility & Credibility**: Pre-registering your sample size (and analysis plan) increases trust in results — especially important when presenting to executives or technical supervisors (see data analysis report structure PDF).

### Rule of thumb
Most online A/B testing platforms and textbooks target **80% power** (β = 0.20) and **α = 0.05** (two-sided) as a reasonable balance between certainty and practicality.


## Theory: When (and When NOT) to Stop an A/B Test

This is one of the most common and costly mistakes in experimentation.

### The Problem with “Peeking” and Early Stopping

- If you repeatedly check the p-value while the test is running and stop as soon as p < 0.05, you **dramatically inflate the false positive rate** (Type I error).
- Even if there is **no real difference**, you will eventually see “significant” results just by chance if you keep peeking.
- Simulations show that with continuous peeking, your actual false positive rate can easily reach 20-30% or higher instead of the nominal 5%.

### Recommended Approach: Fixed Horizon Testing

1. Calculate the required sample size **before** starting (using the methods in this notebook).
2. Run the experiment until you reach (or slightly exceed) that N.
3. **Only then** perform the statistical test and make a decision.
4. Pre-commit to this plan (document it) — this protects you from optional stopping bias.

### Better Alternatives When You Need Flexibility

- **Sequential Testing / Alpha-Spending**: Methods like O’Brien-Fleming boundaries, Sequential Probability Ratio Test (SPRT), or Bayesian posterior probability thresholds. These adjust the significance threshold as data accumulates.
- **Bayesian Methods**: Monitor posterior probability that variant B is better, or expected loss. Often more intuitive for business stakeholders.
- **Platform Features**: Many modern A/B tools (Optimizely, VWO, Google Optimize, etc.) implement “peeking-safe” calculations automatically.

**Golden Rule**: Decide your stopping rule and sample size **before** you launch the experiment. Write it down. This is part of good experimental design and makes your data analysis report trustworthy for all audiences.


## Flowchart: Desired Outcome for A/B Test Sample Size & Execution

This flowchart shows the complete recommended process. It emphasizes **fixed-horizon** testing (run to pre-calculated N) and tailoring the final report to different audiences (as discussed in the provided PDFs on audience analysis and data analysis report structure).

```mermaid
flowchart TD
    A[Start: Define Business / Product Objective<br/>e.g. Increase ad click-through rate] --> B[Identify Baseline Conversion Rate<br/>&amp; Target / Desired Rate]
    B --> C[Calculate MDE<br/>(Minimum Detectable Effect)<br/>Relative lift % = (Target - Baseline) / Baseline × 100]
    C --> D[Choose Statistical Guardrails<br/>Significance Level α e.g. 0.05<br/>Desired Power 1-β e.g. 0.80<br/>Test: Chi-Square or z-test for proportions]
    D --> E[Calculate Required Sample Size N<br/>per variant + total<br/>using Power Analysis]
    E --> F[Estimate Experiment Duration<br/>using expected daily traffic<br/>&amp; % of traffic allocated to test]
    F --> G[Run Experiment to Fixed Sample Size<br/><b>CRITICAL: Do NOT peek or stop early</b><br/>based on interim p-values<br/>(inflates false positives)]
    G --> H[Analyze Results<br/>Chi-Square test, p-value, lift,<br/>confidence interval, practical significance]
    H --> I{Statistically significant<br/>at chosen α ?}
    I -->|Yes + Practical Lift| J[Implement Winner<br/>Roll out new variant]
    I -->|No| K[Do not implement or<br/>iterate hypothesis / increase N]
    J --> L[Document &amp; Report Findings<br/><b>Tailor to Audience:</b><br/>• Executives: Summary, business impact, ROI, recommendation<br/>• Technical/Supervisors: Full methods, diagnostics, code, edge cases<br/>• Mixed: Layered report with clear sections &amp; signposts]
    K --> L
    L --> M[End: Organizational Learning<br/>Update baseline for future tests<br/>Share insights across teams]
    style G fill:#ffcccc,stroke:#cc0000
    style L fill:#e6f3ff,stroke:#0066cc
```

**Key takeaway from flowchart (and PDFs):** The final step is critical — your data analysis report must be adapted to the audience's data literacy, subject knowledge, and time constraints. Executives want the headline and decision; technical reviewers want reproducibility and rigor.


## Expanded Exercise: Planning Sample Size for a New Advertisement Test

You work for an e-commerce company in Costa Rica. Currently, **10%** of people who see your main advertisement click on it and visit the product page. 

Your marketing team has created a **new ad creative**. Internal stakeholders believe it can increase the click-through rate to **at least 14%**.

You decide to run an A/B test (control = old ad, treatment = new ad) and will use a **Chi-Square test** (or equivalent z-test for proportions) to decide whether the difference is statistically significant.

You want the probability of missing a real 4 percentage point lift (if it exists) to be reasonably low — standard practice is to target **80% power**.

### Your tasks (step-by-step):

1. Identify the **baseline conversion rate** and the **significance threshold (α)** from the description.
2. Calculate the **Minimum Detectable Effect (MDE)** — note: it is expressed as a **relative percentage lift**, not an absolute percentage point difference.
3. Use the provided power analysis code (or formula) to calculate the required **total sample size** (the calculator assumes equal allocation: 50% control / 50% treatment).
4. Answer reflection questions about what this sample size means practically.
5. Explore variations in the simulation section by changing inputs.


### Step-by-Step Solution with Explanations

#### Step 1 & 2: Identify inputs and calculate MDE

From the problem:
- Baseline conversion rate = **10%** = 0.10
- Target conversion rate = **14%** = 0.14
- Significance threshold α = **0.05** (standard)

MDE (relative) = (0.14 - 0.10) / 0.10 × 100 = **40%**

We want to detect at least a **40% relative improvement** over the current baseline.


In [ ]:
import numpy as np
from statsmodels.stats.proportion import proportion_effectsize
from statsmodels.stats.power import zt_ind_solve_power

def calculate_sample_size(baseline_rate, mde_relative, alpha=0.05, power=0.80):
    """Return total sample size (control + treatment) for two-proportion z-test."""
    target_rate = baseline_rate * (1 + mde_relative / 100.0)
    es = proportion_effectsize(baseline_rate, target_rate)
    n_per_group = zt_ind_solve_power(
        effect_size=es,
        alpha=alpha,
        power=power,
        ratio=1.0,
        alternative='two-sided'
    )
    total_n = int(np.ceil(n_per_group * 2))
    return total_n, n_per_group

# ========== MAIN EXERCISE SOLUTION ==========
baseline = 0.10
target = 0.14
alpha = 0.05
power = 0.80

MDE = (target - baseline) / baseline * 100
print("Baseline conversion rate:", baseline * 100, "%")
print("Target conversion rate:", target * 100, "%")
print("Minimum Detectable Effect (relative):", round(MDE, 1), "%")
print("Significance level (alpha):", alpha)

total_n, n_per = calculate_sample_size(baseline, MDE, alpha, power)
print("\nTotal sample size needed (both variants):", total_n)
print("Sample size per variant:", n_per)
print("Expected duration example (2000 daily visitors, 50% allocation): ~", int(np.ceil(total_n / 1000)), "days")


#### Alternate Implementation 1: Manual Formula (no external lib beyond numpy/scipy)

Useful when you cannot or do not want to rely on `statsmodels`. The formula below is the classic normal approximation for two proportions (two-sided test).


In [ ]:
from scipy.stats import norm

def calculate_sample_size_manual(baseline_rate, mde_relative, alpha=0.05, power=0.80):
    """Manual formula for total sample size (two-sided, equal groups)."""
    p1 = baseline_rate
    p2 = p1 * (1 + mde_relative / 100.0)
    p_bar = (p1 + p2) / 2
    z_alpha = norm.ppf(1 - alpha / 2)
    z_beta = norm.ppf(power)
    numerator = (z_alpha * np.sqrt(2 * p_bar * (1 - p_bar)) +
                 z_beta * np.sqrt(p1 * (1 - p1) + p2 * (1 - p2)))**2
    denominator = (p2 - p1)**2
    n_per_group = numerator / denominator
    return int(np.ceil(n_per_group * 2)), n_per_group

total_manual, per_manual = calculate_sample_size_manual(baseline, MDE, alpha, power)
print("Manual formula total N:", total_manual, "(per group:", round(per_manual), ")")
print("Difference from statsmodels:", total_manual - total_n, "(very small, normal)")


#### Alternate Implementation 2: Direct effect size + power solve (most transparent)


In [ ]:
# Direct calculation showing the effect size explicitly
es = proportion_effectsize(0.10, 0.14)
print("Cohen's h (effect size for proportions):", round(es, 4))

n_per_direct = zt_ind_solve_power(effect_size=es, alpha=0.05, power=0.80, ratio=1, alternative='two-sided')
print("n per group (direct):", round(n_per_direct))
print("Total N (direct):", int(np.ceil(n_per_direct * 2)))


## Solutions to Additional Practice Exercises


In [ ]:
print("=== Exercise A: Stricter (alpha=0.01, power=0.90) ===")
total_strict, _ = calculate_sample_size(0.10, 40, alpha=0.01, power=0.90)
print("Total N (stricter):", total_strict, "→ much larger as expected")

print("\n=== Exercise B: Lower baseline 5% with same 40% relative MDE ===")
total_low, _ = calculate_sample_size(0.05, 40, alpha=0.05, power=0.80)
print("Total N (baseline 5%):", total_low)
print("Note: Lower baseline usually requires LARGER sample because proportions near 0 or 1 have lower variance, but the relative lift moves it into higher variance region.")


## Simulation Section: Modify Inputs and Re-run to See Impact

This section lets you quickly explore trade-offs. Change any of the four parameters and re-execute the cell. Observe how sample size reacts.

**Key intuitions to develop**:
- Smaller MDE → much larger N (you need more data to reliably detect tiny effects)
- Higher power or lower alpha → larger N
- Baseline near 50% → larger N (maximum variance)
- Very low or high baseline → sometimes smaller N for same relative MDE


In [ ]:
# ========== SIMULATION: Change these four values ==========
baseline_rate = 0.10      # try 0.05, 0.08, 0.20
mde_relative = 40         # try 20, 30, 50, 100
alpha = 0.05              # try 0.01, 0.10
desired_power = 0.80      # try 0.70, 0.90

print("Parameters:")
print(f"  Baseline: {baseline_rate*100:.1f}% | MDE: {mde_relative}% relative | α={alpha} | Power={desired_power*100:.0f}%")

tot, per = calculate_sample_size(baseline_rate, mde_relative, alpha, desired_power)
print(f"\n→ Total sample size needed: {tot}  (≈ {per:.0f} per variant)")

# Quick duration estimate
daily_traffic = 2000
allocation = 0.50
days_needed = int(np.ceil(tot / (daily_traffic * allocation)))
print(f"Estimated days @ {daily_traffic} daily visitors & {allocation*100:.0f}% allocation: {days_needed} days")


## Monte Carlo Verification: Does the Calculated N Actually Give ~80% Power?

Theory is nice; simulation confirms it works. We generate many synthetic A/B tests at the calculated N and count how often the Chi-Square test correctly detects the difference.

This also illustrates why we must run the **full sample size** — early stopping would ruin the power guarantee.


In [ ]:
from scipy.stats import chi2_contingency
import warnings
warnings.filterwarnings('ignore')

def simulate_ab_test(n_per_variant, p_control, p_treatment, n_sim=5000, alpha=0.05):
    """Run many simulated A/B tests and return detection rate (power)."""
    detections = 0
    for _ in range(n_sim):
        # Simulate binomial counts
        clicks_control = np.random.binomial(n_per_variant, p_control)
        clicks_treatment = np.random.binomial(n_per_variant, p_treatment)
        # Chi-Square test on 2x2 table
        table = np.array([[clicks_control, n_per_variant - clicks_control],
                          [clicks_treatment, n_per_variant - clicks_treatment]])
        _, p_val, _, _ = chi2_contingency(table, correction=False)
        if p_val < alpha:
            detections += 1
    return detections / n_sim

# Use the numbers from main exercise
n_per = total_n // 2
empirical_power = simulate_ab_test(n_per, baseline, target, n_sim=3000)
print(f"Empirical power at N={total_n} (n_per={n_per}): {empirical_power*100:.1f}%")
print("(Should be close to the target 80%. Small simulation noise is normal.)")


## Connecting to Audience Analysis & Data Analysis Report Structure

The PDFs provided emphasize that every data product (visualization, report, experiment result) must be adapted to its audience.

After running this A/B test you will produce a **data analysis report**. Different readers need different depths:

- **Primary collaborator / marketing manager** (mixed technical level): Reads Introduction + Conclusion first. Wants the lift, p-value, business recommendation, and estimated ROI. Use clear language, avoid heavy stats jargon.
- **Executive / C-level** (short time span): Skims Introduction + Conclusion for the headline decision. Put the “so what” and recommended action in the first paragraph.
- **Technical supervisor / data science peer** (high data literacy): Reads the Body and Appendix in detail. Needs exact methods, code, assumptions checked, edge cases, and reproducibility notes.

**Best practice**: Write the report in layers. Use headings that let each audience quickly find what they need (see “question-oriented” structure in the paper-structure PDF). Include the sample size justification and stopping rule in the Methods section — this builds enormous credibility.

Example signpost sentence you can adapt:

> “We powered the test to detect a minimum 40% relative lift with 80% power at α = 0.05, requiring 2,060 total participants. The experiment ran to completion without early stopping. Full statistical details and code are in the Appendix.”


In [ ]:
print("\n" + "="*60)
print("SUMMARY OF KEY RESULTS (Main Exercise)")
print("="*60)
print(f"Baseline: {baseline*100}% → Target: {target*100}% (MDE = {MDE:.1f}% relative)")
print(f"Alpha = {alpha} | Power = {power*100}%")
print(f"Required total sample size: {total_n} ({n_per} per variant)")
print("="*60)
print("Remember: Run the FULL sample size. Do not peek and stop early.")
print("Tailor your final report to the specific audience reading it.")
